In [3]:
import truststore
truststore.inject_into_ssl()

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed (free - runs locally)
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

#### RETRIEVAL and GENERATION ####

# Prompt
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
{context}
Question: {question}
""")

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question
print(rag_chain.invoke("What is Task Decomposition?"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9145.85it/s]


According to the context, Task decomposition can be done in several ways, including: 

1. By LLM with simple prompting, 
2. By using task-specific instructions, 
3. With human inputs, or 
4. By relying on an external classical planner (LLM+P approach) that utilizes the Planning Domain Definition Language (PDDL) to describe the planning problem.

However, the context does not provide a direct definition of Task Decomposition. It only explains the different methods of achieving it. Task decomposition can be inferred as the process of breaking down a task into smaller sub-tasks or steps, but this is not explicitly stated in the context.
